In [5]:
import torch
from torch import nn
from torch.nn import functional as F

In [6]:
torch.manual_seed(1337)

batch_size = 64
block_size = 256 # max context length
n_embd = 384
lr = 3e-4
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_iters = 200
eval_interval = 500
max_iters = 5000
dropout = 0.2
n_layer = 6
n_heads = 6

In [7]:
with open(f"dialogues_train.txt", "r", encoding="utf-8") as f:
    text = f.read()

text = text.replace("__eou__ ", "")

chars = sorted(list(set(text)))
vocab_size = len(chars)

s_i = { ch : i for i, ch in enumerate(chars) }
i_s = { i : ch for i, ch in enumerate(chars) }

encode = lambda s: [s_i[c] for c in s]
decode = lambda l: "".join([i_s[i] for i in l])

In [8]:
data = torch.tensor(encode(text), dtype=torch.long)

n = int(0.8*len(data))

dtr = data[:n]
dval = data[n:]

def get_batch(split):
    data = dtr if split == "train" else dval
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])

    x, y = x.to(device), y.to(device)

    return x, y

In [9]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        v = self.value(x)
        out = wei @ v

        return out

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList((Head(head_size) for _ in range(num_heads)))
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))

        return out

In [11]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )


    def forward(self, x):
        return self.net(x)

In [12]:
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.f_fwd = FeedForward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.f_fwd(self.ln2(x))

        return x

In [13]:
class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, n_embd)
        self.position_embd = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(*[Block(n_embd, n_heads) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)


        self.lm_head = nn.Linear(n_embd, vocab_size)


    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embd(idx) # (B, T, C)
        pos_emb = self.position_embd(torch.arange(T, device=device)) # (T, C)

        x = tok_emb + pos_emb # (B, T, C)
        x = self.blocks(x)

        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)

            targets = targets.view(-1)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # (B, C)

            probs = F.softmax(logits, dim=-1) # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)

        return idx

In [14]:
net = Bigram()
n = net.to(device)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3)

In [18]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(net.generate(context, max_new_tokens=100)[0].tolist()))


Good choice , What , I just have to pay Theatrel Text tomorrow . What happened ? What sort of my gir


In [15]:
@torch.no_grad()
def estimate_loss():
    out = {}
    net.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, y = get_batch(split)
            logits, loss = net(X, y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    net.train()
    return out

In [17]:
estimate_loss()

{'train': tensor(0.9370), 'val': tensor(0.9945)}

In [16]:
for step in range(max_iters):
    Xb, yb = get_batch("train")

    logits, loss = net(Xb, yb)

    optimizer.zero_grad(set_to_none=True)

    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(f'{step} / {max_iters}: {loss.item()}')

print("Final loss:", loss)

0 / 5000: 4.881033897399902
500 / 5000: 1.7223924398422241
1000 / 5000: 1.3737765550613403
1500 / 5000: 1.2594274282455444
2000 / 5000: 1.2039151191711426
2500 / 5000: 1.1523966789245605
3000 / 5000: 1.1067767143249512
3500 / 5000: 1.0913455486297607
4000 / 5000: 1.0234320163726807
4500 / 5000: 1.0353667736053467
Final loss: tensor(1.0045, device='cuda:0', grad_fn=<NllLossBackward0>)
